In [2]:
import os
import torch
import numpy as np

In [18]:
repo = 'OxWearables/ssl-wearables'
harnet5 = torch.hub.load(repo, 'harnet5', class_num=29, pretrained=True)
for param in harnet5.feature_extractor.parameters():
    param.requires_grad = False

Using cache found in C:\Users\carol/.cache\torch\hub\OxWearables_ssl-wearables_main
C:\Users\carol/.cache\torch\hub\OxWearables_ssl-wearables_main\hubconf.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues relat

131 Weights loaded


In [4]:
harnet5

Resnet(
  (feature_extractor): Sequential(
    (layer1): Sequential(
      (0): Conv1d(3, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False, padding_mode=circular)
      (1): ResBlock(
        (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False, padding_mode=circular)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False, padding_mode=circular)
        (relu): ReLU(inplace=True)
      )
      (2): ResBlock(
        (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv1): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False, padding_mode=circular)
        (conv2): Conv1d(6

In [6]:
from app_config import RAW_DATA_DIR,RAW_DATA_DIR_TEST,RAW_DATA_DIR_TRAIN
from run_config import SLIDING_WINDOW_LENGTH, SLIDING_WINDOW_STEP, NB_SENSOR_CHANNELS
from utils import data_preprocessing
import sliding_window_on_data
from models.DeepConvLSTM import HARDataset

2024-12-21 09:38:24.953 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition


No GPU available, training on CPU; consider making n_epochs very small.


In [7]:
import numpy as np
import torch
from scipy import signal


# Funzione per ridimensionare le finestre
def resize_windows(X, target_length, target_channels):
    # Riduci i canali da 9 a 3 (puoi fare una media o selezionare specifici canali)
    X_reduced = X[:, :, :target_channels]  # Seleziona i primi 3 canali

    # Ridimensiona la lunghezza della finestra
    X_resized = np.array([signal.resample(window, target_length, axis=0) for window in X_reduced])

    return X_resized

In [13]:
# Prepara i dati
datasetTracesTrain = data_preprocessing.build_dataset(RAW_DATA_DIR_TRAIN)
datasetTracesTest = data_preprocessing.build_dataset(RAW_DATA_DIR_TEST)
dataset_train_labled = data_preprocessing.add_labels_to_dataset(datasetTracesTrain)
dataset_test_labled = data_preprocessing.add_labels_to_dataset(datasetTracesTest)
X_Train, Y_Train = sliding_window_on_data.apply_sliding_window(dataset_train_labled, SLIDING_WINDOW_LENGTH, SLIDING_WINDOW_STEP, NB_SENSOR_CHANNELS)
X_Test, Y_Test = sliding_window_on_data.apply_sliding_window(dataset_test_labled, SLIDING_WINDOW_LENGTH, SLIDING_WINDOW_STEP, NB_SENSOR_CHANNELS)

# Ridimensiona le finestre per adattarle al modello harnet5
X_Train_resized = resize_windows(X_Train, target_length=150, target_channels=3)
X_Test_resized = resize_windows(X_Test, target_length=150, target_channels=3)

print("Shape originale:", X_Test.shape)
print("Shape ridimensionata:", X_Test_resized.shape)

X_Train_resized = X_Train_resized.transpose(0, 2, 1)
X_Test_resized=X_Test_resized.transpose(0,2,1)

print("Shape ridimensionata dopo il traspose:", X_Train_resized.shape)
print("Shape ridimensionata dopo il traspose:", X_Test_resized.shape)



Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial01
Annotations shape: (12024,)
Signals shape: (12024, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial02
Annotations shape: (7214,)
Signals shape: (7214, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity01_Trial03
Annotations shape: (7114,)
Signals shape: (7114, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial01
Annotations shape: (9819,)
Signals shape: (9819, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial02
Annotations shape: (9518,)
Signals shape: (9518, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity02_Trial03
Annotations shape: (10120,)
Signals shape: (10120, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity03_Trial01
Annotations shape: (3707,)
Signals shape: (3707, 11)
Loaded annotations and signals for TraceID: UMAH_User01_Activity03_Trial02
Annotations shape: (5210,

In [19]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

# Converti in torch.FloatTensor
X_Train_tensor = torch.FloatTensor(X_Train_resized)
X_Test_tensor = torch.FloatTensor(X_Test_resized)

# Creazione dataset
train_dataset = HARDataset(X_Train_tensor, Y_Train)
test_dataset = HARDataset(X_Test_tensor, Y_Test)

# Creazione DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Definisci il modello completo utilizzando il classificatore predefinito
class ModelWithFrozenExtractor(nn.Module):
    def __init__(self, model):
        super(ModelWithFrozenExtractor, self).__init__()
        self.feature_extractor = model.feature_extractor
        self.classifier = model.classifier

    def forward(self, x):
        with torch.no_grad():
            features = self.feature_extractor(x)
            features = features.view(features.size(0), -1)
        output = self.classifier(features)
        return output

# Crea il modello completo
model = ModelWithFrozenExtractor(harnet5)

# Definisci la loss e l'ottimizzatore
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)  # Addestra solo il classificatore

# Addestra il modello
num_epochs = 10  # Scegli il numero di epoche
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:  # Assicurati di avere un dataloader per i tuoi dati
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Valuta il modello
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:  # Assicurati di avere un dataloader per i tuoi dati
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy: {:.2f}%'.format(100 * correct / total))

Accuracy: 43.49%
